# 06 - Content-Hash Deduplication and Detection-Run Lineage Demo

This notebook accompanies `06-detection-deduplication-and-model-version-drift.md`.

It implements the proposed `detection_runs` design from Part 4 of that chapter: a content-hash-keyed
lookup that runs **before** the (simulated) Sagemaker/Lambda detection call, deciding whether to
short-circuit a true duplicate, link a corrected resubmission to what it supersedes, attribute a
reprocessing to a model-version change, or run inference normally for a genuinely new document. All
four of the chapter's decision paths are exercised explicitly, end to end.

Extends the `StubChartDetectorModel` pattern from `04_lambda_inference_handler_demo.ipynb` -- fully
offline, only `PIL`, `numpy`, and the standard library (`hashlib`, `uuid`).

In [1]:
import hashlib
import io
import time
from uuid import uuid4

from PIL import Image
import numpy as np

print("Ready.")

Ready.


## 1. A stub detector, same shape as notebook 04's, but counting real inference calls

Tracking `INFERENCE_CALL_COUNT` makes the cost-saving half of the dedup fix ("short-circuit without
paying for another inference call," chapter 06 Part 4) directly measurable rather than just asserted.

In [2]:
INFERENCE_CALL_COUNT = 0


class StubChartDetectorModel:
    """Same stand-in as notebook 04's -- fixed synthetic detections, no real model. Increments a
    module-level counter on every predict() call so the dedup short-circuit's savings are provable."""

    def predict(self, image: Image.Image, confidence_threshold: float = 0.25):
        global INFERENCE_CALL_COUNT
        INFERENCE_CALL_COUNT += 1
        width, height = image.size
        synthetic_detections = [
            {"box": [0.10, 0.15, 0.45, 0.55], "confidence": 0.96, "class": "chart"},
            {"box": [0.55, 0.20, 0.90, 0.50], "confidence": 0.31, "class": "chart"},
        ]
        results = []
        for det in synthetic_detections:
            if det["confidence"] < confidence_threshold:
                continue
            x1, y1, x2, y2 = det["box"]
            results.append({
                "class": det["class"],
                "confidence": det["confidence"],
                "bbox_pixels": [round(x1 * width), round(y1 * height), round(x2 * width), round(y2 * height)],
            })
        return results


_MODEL = StubChartDetectorModel()
print("Stub model ready. Inference calls so far:", INFERENCE_CALL_COUNT)

Stub model ready. Inference calls so far: 0


## 2. The `detection_runs` store -- an in-memory stand-in for the proposed DynamoDB table

Chapter 06 Part 4 places this lookup in the ECS Fargate orchestration layer (stateful) rather than the
Lambda handler itself (stateless, per Chapter 04's argument) -- exactly the "hold state across
invocations" role a DynamoDB table would play in production. Indexed two ways, matching the two lookups
the orchestration layer needs: by `content_hash` (is this exact content new) and by `document_id` (has
this logical document been processed under different bytes before).

In [3]:
class DetectionRunStore:
    """In-memory stand-in for a DynamoDB detection_runs table, indexed by content_hash and
    document_id -- exactly the two lookups chapter 06 Part 4 describes the orchestration layer doing
    before ever invoking the detection endpoint."""

    def __init__(self):
        self.runs: dict[str, dict] = {}                 # detection_run_id -> run record
        self._by_hash: dict[str, list[str]] = {}         # content_hash -> [detection_run_id, ...]
        self._by_document: dict[str, list[str]] = {}     # document_id -> [detection_run_id, ...]

    def find_by_hash(self, content_hash: str) -> list[dict]:
        return [self.runs[rid] for rid in self._by_hash.get(content_hash, [])]

    def find_latest_by_document(self, document_id: str) -> dict | None:
        run_ids = self._by_document.get(document_id, [])
        if not run_ids:
            return None
        # "latest" = most recently created run for this document that hasn't itself been superseded
        candidates = [self.runs[rid] for rid in run_ids]
        current = [r for r in candidates if not any(
            other["supersedes_run_id"] == r["detection_run_id"] for other in candidates
        )]
        return max(current, key=lambda r: r["created_at"]) if current else None

    def record(self, *, content_hash: str, document_id: str, model_version: str,
               detections: list, supersedes_run_id: str | None, client: str) -> dict:
        run = {
            "detection_run_id": str(uuid4()),
            "document_id": document_id,
            "content_hash": content_hash,
            "model_version": model_version,
            "supersedes_run_id": supersedes_run_id,
            "client": client,
            "detections": detections,
            "created_at": time.time(),
        }
        self.runs[run["detection_run_id"]] = run
        self._by_hash.setdefault(content_hash, []).append(run["detection_run_id"])
        self._by_document.setdefault(document_id, []).append(run["detection_run_id"])
        return run


store = DetectionRunStore()
print("detection_runs store ready.")

detection_runs store ready.


## 3. `content_hash` -- SHA-256 of the raw image bytes, computed before any inference call

Mirrors chapter 06 Part 4's `content_hash = sha256(raw_image_bytes)`. Note this hashes the **raw
bytes**, not a decoded/re-encoded PIL image -- byte-identical resubmissions must hash identically
regardless of what happens further down the pipeline.

In [4]:
def content_hash(raw_bytes: bytes) -> str:
    return hashlib.sha256(raw_bytes).hexdigest()


def make_synthetic_image_bytes(seed: int, size=(400, 300)) -> bytes:
    """Deterministic synthetic image bytes, so the same seed always produces byte-identical PNG
    output -- standing in for a real scanned/exported document page."""
    arr = (np.random.default_rng(seed).random((size[1], size[0], 3)) * 255).astype(np.uint8)
    buf = io.BytesIO()
    Image.fromarray(arr).save(buf, format="PNG")
    return buf.getvalue()


doc_v1_bytes = make_synthetic_image_bytes(seed=1)
print("content_hash(doc_v1_bytes) =", content_hash(doc_v1_bytes)[:16], "...")
print("Same bytes hash identically:", content_hash(doc_v1_bytes) == content_hash(doc_v1_bytes))

content_hash(doc_v1_bytes) = aa0c6b57e0e7a2f4 ...
Same bytes hash identically: True


## 4. `process_document` -- the four-path decision logic from chapter 06 Part 4

Implements the lookup-before-inference sequence exactly as described: check `content_hash` first: if
found under the **same** `model_version`, short-circuit and return the cached detections with **no
inference call**. If found under a **different** model version, re-run inference (the retrain
legitimately requires a new run) and link it via `supersedes_run_id`. If not found by hash, but
`document_id` has a prior run, treat it as a corrected resubmission -- re-run and supersede. Otherwise,
a genuinely new document -- run normally.

In [5]:
def process_document(raw_bytes: bytes, document_id: str, model_version: str, client: str,
                      confidence_threshold: float = 0.25) -> dict:
    h = content_hash(raw_bytes)

    # Path 1: exact content already processed under this exact model -- true duplicate.
    hash_matches = store.find_by_hash(h)
    same_model_match = next((r for r in hash_matches if r["model_version"] == model_version), None)
    if same_model_match:
        return {**same_model_match, "path": "DUPLICATE_SHORT_CIRCUIT", "inference_ran": False}

    # Path 2: exact content processed before, but under a DIFFERENT model version -- re-run, supersede.
    other_model_match = next((r for r in hash_matches if r["model_version"] != model_version), None)
    if other_model_match:
        image = Image.open(io.BytesIO(raw_bytes)).convert("RGB")
        detections = _MODEL.predict(image, confidence_threshold=confidence_threshold)
        run = store.record(content_hash=h, document_id=document_id, model_version=model_version,
                            detections=detections, supersedes_run_id=other_model_match["detection_run_id"],
                            client=client)
        return {**run, "path": "MODEL_VERSION_REPROCESS", "inference_ran": True}

    # Path 3: content hash not seen before, but this document_id has a prior run -- corrected resubmission.
    prior_for_document = store.find_latest_by_document(document_id)
    if prior_for_document is not None:
        image = Image.open(io.BytesIO(raw_bytes)).convert("RGB")
        detections = _MODEL.predict(image, confidence_threshold=confidence_threshold)
        run = store.record(content_hash=h, document_id=document_id, model_version=model_version,
                            detections=detections, supersedes_run_id=prior_for_document["detection_run_id"],
                            client=client)
        return {**run, "path": "CORRECTED_RESUBMISSION", "inference_ran": True}

    # Path 4: genuinely new document -- run inference normally, nothing to supersede.
    image = Image.open(io.BytesIO(raw_bytes)).convert("RGB")
    detections = _MODEL.predict(image, confidence_threshold=confidence_threshold)
    run = store.record(content_hash=h, document_id=document_id, model_version=model_version,
                        detections=detections, supersedes_run_id=None, client=client)
    return {**run, "path": "NEW_DOCUMENT", "inference_ran": True}


print("process_document defined.")

process_document defined.


## 5. Path 4 then Path 1: a genuinely new document, then an exact-duplicate resubmission

The classic "an upstream retry fired twice" scenario from chapter 06 Part 1. The second call must not
increment `INFERENCE_CALL_COUNT` at all.

In [6]:
before = INFERENCE_CALL_COUNT
result_1 = process_document(doc_v1_bytes, document_id="doc-alpha", model_version="yolov5-2026-03-01",
                             client="eli-lilly")
print("Run 1:", result_1["path"], "| inference_ran:", result_1["inference_ran"])
assert result_1["path"] == "NEW_DOCUMENT" and result_1["inference_ran"]
assert INFERENCE_CALL_COUNT == before + 1

# Exact same bytes submitted again -- upstream retry, or a client resending the same file.
result_2 = process_document(doc_v1_bytes, document_id="doc-alpha", model_version="yolov5-2026-03-01",
                             client="eli-lilly")
print("Run 2 (exact resubmission):", result_2["path"], "| inference_ran:", result_2["inference_ran"])
assert result_2["path"] == "DUPLICATE_SHORT_CIRCUIT" and not result_2["inference_ran"]
assert result_2["detection_run_id"] == result_1["detection_run_id"], "must return the SAME cached run"
assert INFERENCE_CALL_COUNT == before + 1, "the duplicate must not have paid for a second inference call"

print("\nConfirmed: the exact duplicate short-circuited to the cached run with zero additional")
print(f"inference calls (total calls still {INFERENCE_CALL_COUNT}).")

Run 1: NEW_DOCUMENT | inference_ran: True
Run 2 (exact resubmission): DUPLICATE_SHORT_CIRCUIT | inference_ran: False

Confirmed: the exact duplicate short-circuited to the cached run with zero additional
inference calls (total calls still 1).


## 6. Path 3: a corrected resubmission -- different bytes, same logical document

The client fixes a typo on the page and re-exports it. Different content hash, same `document_id` --
this must re-run inference (the content genuinely changed) and link the new run to what it supersedes,
rather than sitting as an unrelated duplicate the way Part 1 describes today's behavior.

In [7]:
doc_v1_corrected_bytes = make_synthetic_image_bytes(seed=2)  # different bytes, same logical document
assert content_hash(doc_v1_corrected_bytes) != content_hash(doc_v1_bytes)

result_3 = process_document(doc_v1_corrected_bytes, document_id="doc-alpha",
                             model_version="yolov5-2026-03-01", client="eli-lilly")
print("Run 3 (corrected resubmission):", result_3["path"], "| inference_ran:", result_3["inference_ran"])
assert result_3["path"] == "CORRECTED_RESUBMISSION" and result_3["inference_ran"]
assert result_3["supersedes_run_id"] == result_1["detection_run_id"]

# The store's "latest for this document" lookup must now point at the corrected run, not the original.
latest = store.find_latest_by_document("doc-alpha")
assert latest["detection_run_id"] == result_3["detection_run_id"]
print("\nConfirmed: the corrected resubmission re-ran inference and is now the current run for")
print("doc-alpha -- the original run is preserved (queryable by id) but no longer 'latest.'")

Run 3 (corrected resubmission): CORRECTED_RESUBMISSION | inference_ran: True

Confirmed: the corrected resubmission re-ran inference and is now the current run for
doc-alpha -- the original run is preserved (queryable by id) but no longer 'latest.'


## 7. Path 2: the same content, reprocessed after a model retrain

The retraining pipeline (Chapter 02) promotes a new fine-tune. The exact same bytes come through again
-- content hash matches an existing run, but under a different `model_version`. This is a legitimate
reprocess, not a duplicate: re-run inference, and link the new run via `supersedes_run_id` so both
remain queryable and attributable to their respective model version.

In [8]:
result_4 = process_document(doc_v1_corrected_bytes, document_id="doc-alpha",
                             model_version="yolov5-2026-06-14", client="eli-lilly")
print("Run 4 (same content, new model version):", result_4["path"], "| inference_ran:", result_4["inference_ran"])
assert result_4["path"] == "MODEL_VERSION_REPROCESS" and result_4["inference_ran"]
assert result_4["supersedes_run_id"] == result_3["detection_run_id"]
assert result_4["model_version"] != result_3["model_version"]

print("\nAll four detection_runs so far:")
for rid, run in store.runs.items():
    supersedes_display = run["supersedes_run_id"][:8] if run["supersedes_run_id"] else "None"
    print(f"  {run['detection_run_id'][:8]}  model={run['model_version']:20s} "
          f"supersedes={supersedes_display:10s} doc={run['document_id']}")

Run 4 (same content, new model version): MODEL_VERSION_REPROCESS | inference_ran: True

All four detection_runs so far:
  25266914  model=yolov5-2026-03-01    supersedes=None       doc=doc-alpha
  539c6aa0  model=yolov5-2026-03-01    supersedes=25266914   doc=doc-alpha
  fcd4f372  model=yolov5-2026-06-14    supersedes=539c6aa0   doc=doc-alpha


## 8. The downstream-reporting tie-in: excluding superseded runs from a per-client count

Chapter 06 Part 5's point made concrete: a monthly detection-count report must walk `supersedes_run_id`
and count only *current* (non-superseded) runs, or a corrected resubmission inflates the client's
document-volume count even though the fix above stopped it from being a totally untracked duplicate.

In [9]:
def current_runs_for_client(client: str) -> list[dict]:
    """Excludes any run that some other run's supersedes_run_id points at -- i.e., only runs that are
    still the latest, current representation of their document_id."""
    all_client_runs = [r for r in store.runs.values() if r["client"] == client]
    superseded_ids = {r["supersedes_run_id"] for r in all_client_runs if r["supersedes_run_id"]}
    return [r for r in all_client_runs if r["detection_run_id"] not in superseded_ids]


current = current_runs_for_client("eli-lilly")
print(f"Total detection_runs recorded for eli-lilly: {len(store.runs)}")
print(f"Current (non-superseded) runs counted in the monthly report: {len(current)}")

# Only ONE document (doc-alpha) was ever really submitted, corrected once, and reprocessed once under
# a new model -- the report must count it as ONE current document, not four.
assert len(current) == 1, "the report must not double-count superseded lineage as separate documents"
print("\nConfirmed: despite four detection_runs existing for audit/lineage purposes, the report layer")
print("correctly counts exactly one current document for doc-alpha.")

Total detection_runs recorded for eli-lilly: 3
Current (non-superseded) runs counted in the monthly report: 1

Confirmed: despite four detection_runs existing for audit/lineage purposes, the report layer
correctly counts exactly one current document for doc-alpha.


## Takeaways

- **The content-hash check runs before the inference call, not after** -- that ordering is what turns
  dedup into an actual cost saving (Section 5), not just a bookkeeping cleanup applied retroactively.
- **Deduplication and supersession are related but distinct outcomes of the same lookup.** An exact
  byte match short-circuits with zero new inference; a changed-bytes-same-document match still requires
  inference, just links the result rather than leaving it orphaned (Sections 5-6) -- exactly chapter
  06 Part 3's Concern A.
- **A model-version change on identical content is neither a duplicate nor a resubmission** -- it is its
  own third path, re-running inference and superseding the prior model version's result, so a retrain's
  effect on a given document stays attributable (Section 7) -- chapter 06 Part 3's Concern B.
- **The lineage the store preserves (all four runs, `supersedes_run_id` chains) has to be actively
  consumed downstream**, not just recorded -- a reporting layer that doesn't walk `supersedes_run_id`
  still overcounts, even though the correct data already exists upstream (Section 8).